In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

In [2]:


def extract_elan_data(eaf_filepath, output_csv_path):
    print(f"正在解析文件: {eaf_filepath} ...")
    
    # 1. 解析 XML 树结构
    tree = ET.parse(eaf_filepath)
    root = tree.getroot()
    
    # 2. 提取“时间槽 (Time Slots)”词典
    # ELAN 的时间不是直接写在标签里的，而是用一个 ID (比如 ts1, ts2) 代替
    # 所以我们先要建一个字典，把 ts1 映射到具体的毫秒数
    time_slots = {}
    time_order = root.find('TIME_ORDER')
    if time_order is not None:
        for slot in time_order.findall('TIME_SLOT'):
            ts_id = slot.get('TIME_SLOT_ID')
            ts_val = slot.get('TIME_VALUE')
            if ts_id and ts_val:
                time_slots[ts_id] = int(ts_val)
                
    # 3. 提取所有的标注内容 (Annotations)
    annotations = []
    for tier in root.findall('TIER'):
        tier_id = tier.get('TIER_ID') # 比如 'Block', 'Stage', 'Event'
        
        for ann in tier.findall('ANNOTATION'):
            align_ann = ann.find('ALIGNABLE_ANNOTATION')
            if align_ann is not None:
                ts1 = align_ann.get('TIME_SLOT_REF1') # 起始时间 ID
                ts2 = align_ann.get('TIME_SLOT_REF2') # 结束时间 ID
                ann_val = align_ann.find('ANNOTATION_VALUE') # 具体的文本
                
                # 如果这个标签有完整的时间和文字
                if ts1 in time_slots and ts2 in time_slots and ann_val is not None:
                    text = ann_val.text.strip() if ann_val.text else ""
                    
                    if text: # 忽略不小心打上的空标签
                        annotations.append({
                            'Tier': tier_id,
                            'Start_ms': time_slots[ts1],
                            'End_ms': time_slots[ts2],
                            'Duration_ms': time_slots[ts2] - time_slots[ts1],
                            'Text': text
                        })
    
    # 4. 转换为 Pandas DataFrame 并按时间排序
    df = pd.DataFrame(annotations)
    df = df.sort_values(by='Start_ms').reset_index(drop=True)
    
    # 5. 导出为 CSV
    df.to_csv(output_csv_path, index=False)
    
    print(f"提取成功！共找到 {len(df)} 个有效标签。")
    print(f"数据已保存至: {output_csv_path}")
    
    return df

# ==========================================
# 运行提取逻辑
# ==========================================
# 替换为你实际的文件名
eaf_file = '001_Elan.eaf'  
csv_output = 'CVC_0001_Annotations.csv'

# 提取并打印前 10 行看看长什么样
df_anno = extract_elan_data(eaf_file, csv_output)
print("\n--- 提取数据预览 (前10行) ---")
print(df_anno.head(10))

正在解析文件: 001_Elan.eaf ...
提取成功！共找到 106 个有效标签。
数据已保存至: CVC_0001_Annotations.csv

--- 提取数据预览 (前10行) ---
        Tier  Start_ms   End_ms  Duration_ms                              Text
0      Block    557142   586485        29343             Experimental Baseline
1  Condition    636436  1105435       468999                              LWBE
2      Block    636456   665907        29451                   LWBE-A-Baseline
3      Block    665907   714601        48694                  LWBE-A-Procedure
4      Stage    670617   679107         8490              Ultrasound  scanning
5      Stage    681993   685652         3659              Ultrasound  scanning
6      Stage    688568   694915         6347              Ultrasound  scanning
7      Stage    696889   704925         8036              Ultrasound  scanning
8      Event    703639   706535         2896                 Unsure of anatomy
9      Block    714601   744142        29541  LWBE-A-Recovery/ LWBE-B-Baseline


In [3]:
#import pandas as pd

# 1. 读取你刚才提取出来的 106 个标签
df = pd.read_csv('CVC_0001_Annotations.csv')

# 2. 核心过滤逻辑：只保留 Tier 为 'Stage' 或 'Event' 的行
# 这样就把那些没用的 Block 和 Condition 全踢掉了
filtered_df = df[df['Tier'].isin(['Stage', 'Event'])].copy()

# 3. 打印出来看看剩下多少个精华标签
print(f"过滤后剩余核心标签数量: {len(filtered_df)}")
print(filtered_df[['Start_ms', 'End_ms', 'Tier', 'Text']])

# 你也可以把它单独存成一个精华版 CSV
# filtered_df.to_csv('CVC_0001_Filtered_Stages.csv', index=False)

过滤后剩余核心标签数量: 81
     Start_ms   End_ms   Tier                   Text
4      670617   679107  Stage   Ultrasound  scanning
5      681993   685652  Stage   Ultrasound  scanning
6      688568   694915  Stage   Ultrasound  scanning
7      696889   704925  Stage   Ultrasound  scanning
8      703639   706535  Event      Unsure of anatomy
..        ...      ...    ...                    ...
100   2661610  2671868  Event                  Bleep
101   2672262  2679921  Event        ODP inattention
102   2689910  2697000  Stage           Wire removal
103   2703495  2709901  Event  Medical student query
105   2710251  2720459  Event         Clinical query

[81 rows x 4 columns]


In [4]:
# 统计一下这 81 个标签里，每个动作出现了多少次
print("--- 动作词频统计 ---")
print(filtered_df['Text'].value_counts())

--- 动作词频统计 ---
Text
Medical student query                                                 13
Bleep                                                                 11
Clinical query                                                        11
Venipuncture                                                           6
Position check (ultrasound)                                            5
Ultrasound scanning                                                    5
ODP inattention                                                        5
Catheterisation                                                        4
Ultrasound  scanning                                                   4
Wire Advancement                                                       3
Postion check (ultrasound)                                             2
Wire removal                                                           2
Other: Not happy with positioning; wire left and ended.                1
Wire Removal  (with catheter)  

In [5]:
#import pandas as pd

# 1. 读取并进行基础清洗（去多余空格、统一首字母大写）
df = pd.read_csv('CVC_0001_Annotations.csv')
filtered_df = df[df['Tier'].isin(['Stage', 'Event'])].copy()

filtered_df['Clean_Text'] = filtered_df['Text'].str.strip().str.replace(r'\s+', ' ', regex=True)
filtered_df['Clean_Text'] = filtered_df['Clean_Text'].str.capitalize()

# 纠正手滑的拼写错误和合并同类项
correction_dict = {
    'Postion check (ultrasound)': 'Position check (ultrasound)',
    'Clinical query odp': 'Clinical query',
    'Wire removal (with catheter)': 'Wire removal',
    'Wire in and minimal advancement': 'Wire advancement',
    'Other: guidewire identified as leaving ij - not happy with anatomy': 'Guidewire identified as leaving ij'
}
filtered_df['Clean_Text'] = filtered_df['Clean_Text'].replace(correction_dict)

# ==========================================
# 2. 核心筛选逻辑：划分【核心步骤】与【环境干扰】
# ==========================================
# 我们把真正属于 CVC 手术操作的词放入白名单
core_whitelist = [
    'Venipuncture', 
    'Position check (ultrasound)', 
    'Ultrasound scanning', 
    'Catheterisation', 
    'Wire advancement', 
    'Wire removal', 
    'Dilator', 
    'Guidewire identified as leaving ij', 
    'Unsure of anatomy', 
    'Ectopics'
]

# 筛选 A：属于核心手术步骤和突发事件的数据
df_core_stages = filtered_df[filtered_df['Clean_Text'].isin(core_whitelist)].copy()

# 筛选 B：属于外部环境干扰的数据（排除掉白名单剩下的就是干扰项，如 Bleep, Query 等）
df_interruptions = filtered_df[~filtered_df['Clean_Text'].isin(core_whitelist)].copy()

# ==========================================
# 3. 打印结果到控制台 (Console)
# ==========================================
print("==================================================")
print(f"Statistics: Out of 81 total tags, filtered {len(df_core_stages)} core steps and {len(df_interruptions)} environmental distractions.")
print("==================================================")

print("\n📊 【Core Surgical Steps & Critical Events】Frequency Count:")
print(df_core_stages['Clean_Text'].value_counts())

print("\n🔔 【External Environmental Interruptions】Frequency Count:")
print(df_interruptions['Clean_Text'].value_counts())

# ==========================================
# 4. 写入 CSV 文件
# ==========================================
core_csv_path = 'CVC_0001_Core_Stages.csv'
interruption_csv_path = 'CVC_0001_Environmental_Interruptions.csv'

df_core_stages.to_csv(core_csv_path, index=False)
df_interruptions.to_csv(interruption_csv_path, index=False)

print("\n==================================================")
print(f"💾 文件写入成功！")
print(f"1. 核心步骤已写入: {core_csv_path}")
print(f"2. 干扰项已写入: {interruption_csv_path}")
print("==================================================")

Statistics: Out of 81 total tags, filtered 39 core steps and 42 environmental distractions.

📊 【Core Surgical Steps & Critical Events】Frequency Count:
Clean_Text
Ultrasound scanning                   9
Position check (ultrasound)           7
Venipuncture                          6
Wire advancement                      5
Catheterisation                       4
Wire removal                          4
Unsure of anatomy                     1
Guidewire identified as leaving ij    1
Dilator                               1
Ectopics                              1
Name: count, dtype: int64

🔔 【External Environmental Interruptions】Frequency Count:
Clean_Text
Medical student query                                      13
Clinical query                                             12
Bleep                                                      11
Odp inattention                                             5
Other: not happy with positioning; wire left and ended.     1
Name: count, dtype: int64

💾 文件写入